# 01. Data collection EMSC

Input: API EMSC. Output: JSON bulanan, events.csv, dan collection_manifest.json. Rentang utama 2021-2025 ditambah testing 2026 hingga batas tanggal eksplisit.

**Lihat hasil:** [jumlah catatan dan contoh data](#s4) | [jumlah per tahun](#s6). Hasil ada di bawah sel kode pada bagian tersebut.

[Panduan](../00_panduan.ipynb) | [Sebelumnya](../00_panduan.ipynb) | [Berikutnya](../02_data_preprocessing/02_data_preprocessing.ipynb)

### Daftar bagian

- [1. Inisialisasi dan dependensi](#s1)
- [2. Definisi atribut dan kedalaman](#s2)
- [3. Pengunduhan bulanan, retry, cache, dan batas API](#s3)
- [4. Jalankan collection dan lihat jumlah data](#s4)
- [5. Audit depth dan cakupan](#s5)
- [6. Jumlah per tahun, jenis catatan, dan rincian bulanan](#s6)

<a id="s1"></a>
## 1. Inisialisasi dan dependensi

Jalankan dari atas ke bawah dengan kernel Python 3.12.11 (Seismic Research). Konfigurasi tersentral di config.json.

In [1]:
from pathlib import Path
import os
import sys

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "2")
os.environ.setdefault("OMP_NUM_THREADS", "2")
candidates = []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / "seismic_research_2021_2025"])
ROOT = next((p for p in candidates if (p / "config.json").is_file() and (p / "src/common.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Buka notebook dari folder proyek seismic_research_2021_2025.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, Image, Markdown, FileLink
from src.common import read_config
cfg = read_config(ROOT)
print("Interpreter:", sys.executable)
print("Python:", sys.version.split()[0])
print("Proyek:", ROOT)
print("Training:", cfg["train_start"], "sampai sebelum", cfg["train_end_exclusive"])
print("Testing sampai sebelum (UTC):", cfg["test_end_exclusive"])

import json
import time
from pathlib import Path
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from src.common import ROOT, file_hash, read_config, write_json

Interpreter: C:\Users\yudhi\.local\bin\python3.12.exe
Python: 3.12.11
Proyek: C:\Users\yudhi\Documents\machine-learning\seismic_research_2021_2025
Training: 2021-01-01 sampai sebelum 2026-01-01
Testing sampai sebelum (UTC): 2026-10-26


<a id="s2"></a>
## 2. Definisi atribut dan kedalaman

Kedalaman berasal dari **properties.depth (km)**. geometry.coordinates[2] disimpan sebagai geometry_z untuk audit; tidak dijadikan kedalaman. Katalog mencakup bounding box Indonesia dan sekitarnya, bukan batas administratif negara.

In [2]:
def parse_feature(feature):
    properties = feature.get("properties") or {}
    coordinates = (feature.get("geometry") or {}).get("coordinates") or []
    return {
        "eventid": feature.get("id") or properties.get("unid"),
        "time": properties.get("time"),
        "longitude": coordinates[0] if len(coordinates) > 0 else None,
        "latitude": coordinates[1] if len(coordinates) > 1 else None,
        "depth": properties.get("depth"),
        "depth_source": "properties.depth",
        "geometry_z": coordinates[2] if len(coordinates) > 2 else None,
        "magnitude": properties.get("mag"),
        "magnitude_type": properties.get("magtype"),
        "event_type": properties.get("evtype"),
        "region": properties.get("flynn_region") or properties.get("region"),
        "lastupdate": properties.get("lastupdate"),
        "source_catalog": properties.get("source_catalog"),
        "author": properties.get("auth"),
    }

<a id="s3"></a>
## 3. Pengunduhan bulanan, retry, cache, dan batas API

Setiap bulan disimpan agar pengulangan tidak mengunduh ulang. Jika respons mencapai limit, interval dipecah hingga tidak terpotong. Kegagalan menghentikan proses; tidak dianggap nol kejadian. Endpoint menerima waktu UTC lengkap. Data disaring lagi secara lokal pada preprocessing.

In [3]:
def collect(root=ROOT):
    root = Path(root)
    cfg = read_config(root)
    start = pd.Timestamp(cfg["train_start"], tz="UTC")
    end = pd.Timestamp(cfg["test_end_exclusive"], tz="UTC")
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=Retry(
        total=4, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"], respect_retry_after_header=True)))
    box = cfg["bbox"]
    base = {"format": "json", "minlatitude": box["min_lat"], "maxlatitude": box["max_lat"],
            "minlongitude": box["min_lon"], "maxlongitude": box["max_lon"],
            "limit": cfg["api_limit"], "orderby": "time-asc"}

    def fetch_interval(a, b):
        params = {**base, "starttime": a.strftime("%Y-%m-%dT%H:%M:%S.%f"),
                  "endtime": (b - pd.Timedelta(microseconds=1)).strftime("%Y-%m-%dT%H:%M:%S.%f")}
        response = session.get(cfg["api_url"], params=params, timeout=(15, 120))
        time.sleep(cfg["request_pause_seconds"])
        if response.status_code == 204:
            return [], [{"url": response.url, "status": 204, "count": 0}]
        response.raise_for_status()
        payload = response.json()
        if payload.get("type") != "FeatureCollection" or not isinstance(payload.get("features"), list):
            raise ValueError("Respons EMSC bukan FeatureCollection; unduhan tidak dinyatakan lengkap.")
        features = payload["features"]
        receipt = {"url": response.url, "status": response.status_code, "count": len(features)}
        if len(features) >= cfg["api_limit"]:
            if b - a <= pd.Timedelta(seconds=1):
                raise RuntimeError("Batas API tercapai bahkan pada interval 1 detik.")
            midpoint = a + (b - a) / 2
            left, lr = fetch_interval(a, midpoint)
            right, rr = fetch_interval(midpoint, b)
            return left + right, [receipt] + lr + rr
        return features, [receipt]

    rows, manifests = [], []
    for a in pd.date_range(start, end, freq="MS", inclusive="left"):
        b = min(a + pd.offsets.MonthBegin(1), end)
        cache = root / f"data/raw/monthly/{a:%Y-%m}.json"
        key = {"start": a.isoformat(), "end_exclusive": b.isoformat(), "base": base, "api": cfg["api_url"]}
        saved = json.loads(cache.read_text()) if cache.exists() else None
        if not saved or saved.get("request") != key or not saved.get("complete"):
            features, receipts = fetch_interval(a, b)
            saved = {"request": key, "complete": True, "retrieved_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
                     "requests": receipts, "features": features}
            write_json(cache, saved)
        records = [parse_feature(f) for f in saved["features"]]
        rows.extend(records)
        manifests.append({"start": a.isoformat(), "end_exclusive": b.isoformat(), "complete": True,
                          "events_returned": len(records), "cache_sha256": file_hash(cache),
                          "retrieved_at_utc": saved["retrieved_at_utc"]})
        print(f"{a:%Y-%m}: {len(records):,} event (cache tersedia)", flush=True)
    if not rows:
        raise ValueError("Katalog kosong. Periksa permintaan; jangan lanjutkan model.")
    events = pd.DataFrame(rows)
    events.to_csv(root / "data/raw/events.csv", index=False)
    write_json(root / "data/raw/collection_manifest.json", {"source": cfg["api_url"], "intervals": manifests,
               "start": start.isoformat(), "end_exclusive": end.isoformat(), "all_complete": True,
               "license": "EMSC / SeismicPortal, CC BY 4.0", "raw_sha256": file_hash(root / "data/raw/events.csv")})
    return events, pd.DataFrame(manifests)

<a id="s4"></a>
## 4. Jalankan collection dan lihat jumlah data

Output sel ini menampilkan total catatan, pembagian training/testing, dan 10 contoh data. Unduhan baru membutuhkan internet; cache yang lengkap digunakan kembali.

In [4]:
from contextlib import redirect_stdout
from io import StringIO

collection_log = StringIO()
with redirect_stdout(collection_log):
    raw, coverage = collect(ROOT)
jumlah = lambda nilai: f"{int(nilai):,}".replace(",", ".")
waktu = pd.to_datetime(raw["time"], utc=True, format="mixed", errors="coerce")
historis = waktu.lt(pd.Timestamp(cfg["train_end_exclusive"], tz="UTC"))
display(Markdown(f"""### Hasil pengambilan data

**Berhasil mengambil {jumlah(len(raw))} catatan dari katalog EMSC.**

| Periode | Jumlah catatan mentah |
| --- | ---: |
| 2021-2025, calon data training | {jumlah(historis.sum())} |
| 2026, calon data testing | {jumlah((~historis & waktu.notna()).sum())} |
| Total | {jumlah(len(raw))} |

Seluruh **{jumlah(len(coverage))} interval bulanan** berhasil diambil.
Batas akhir pengambilan: sebelum **{cfg['test_end_exclusive']} pukul 00.00 UTC**.
Tahun 2026 belum satu tahun penuh.

Jumlah ini masih **catatan mentah**, belum jumlah gempa final untuk analisis.
Jenis kejadian, duplikat, dan batas magnitudo diperiksa pada notebook 02.
Satu baris tabel berikut mewakili satu catatan kejadian, bukan satu kotak peta.
"""))
display(raw[["eventid", "time", "latitude", "longitude", "magnitude", "depth", "event_type", "region"]].head(10).rename(columns={
    "eventid": "ID kejadian", "time": "Waktu UTC", "latitude": "Lintang", "longitude": "Bujur",
    "magnitude": "Magnitudo", "depth": "Kedalaman (km)", "event_type": "Kode jenis", "region": "Wilayah"}))
display(FileLink(str(ROOT / "data/raw/events.csv"), result_html_prefix="Seluruh data mentah: "))


### Hasil pengambilan data

**Berhasil mengambil 78.265 catatan dari katalog EMSC.**

| Periode | Jumlah catatan mentah |
| --- | ---: |
| 2021-2025, calon data training | 50.624 |
| 2026, calon data testing | 27.641 |
| Total | 78.265 |

Seluruh **70 interval bulanan** berhasil diambil.
Batas akhir pengambilan: sebelum **2026-10-26 pukul 00.00 UTC**.
Tahun 2026 belum satu tahun penuh.

Jumlah ini masih **catatan mentah**, belum jumlah gempa final untuk analisis.
Jenis kejadian, duplikat, dan batas magnitudo diperiksa pada notebook 02.
Satu baris tabel berikut mewakili satu catatan kejadian, bukan satu kotak peta.


,ID kejadian,Waktu UTC,Lintang,Bujur,Magnitudo,Kedalaman (km),Kode jenis,Wilayah
0,20210101_0000008,2021-01-01T00:22:54.0Z,-8.20,117.80,2.5,16.0,ke,"SUMBAWA REGION, INDONESIA"
1,20210101_0000007,2021-01-01T00:33:29.0Z,-9.02,118.49,4.3,66.0,ke,"SUMBAWA REGION, INDONESIA"
2,20210101_0000028,2021-01-01T02:33:53.0Z,-1.81,120.57,3.9,10.0,ke,"SULAWESI, INDONESIA"
3,20210101_0000036,2021-01-01T03:21:13.0Z,-10.10,120.78,3.1,73.0,ke,"SUMBA REGION, INDONESIA"
4,20210101_0000035,2021-01-01T03:27:02.0Z,4.29,96.47,2.7,10.0,ke,"NORTHERN SUMATRA, INDONESIA"
5,20210101_0000257,2021-01-01T03:44:06.0Z,-3.05,131.89,3.6,34.0,ke,"CERAM SEA, INDONESIA"
6,20210101_0000045,2021-01-01T04:05:16.6Z,-0.19,124.46,4.6,55.0,ke,MOLUCCA SEA
7,20210101_0000055,2021-01-01T05:03:10.0Z,-9.18,119.05,3.2,46.0,ke,"SUMBA REGION, INDONESIA"
8,20210101_0000064,2021-01-01T05:51:06.0Z,-9.68,119.84,2.8,48.0,ke,"SUMBA REGION, INDONESIA"
9,20210101_0000061,2021-01-01T05:52:29.0Z,-2.77,130.50,3.3,10.0,ke,"CERAM SEA, INDONESIA"


C:\Users\yudhi\Documents\machine-learning\seismic_research_2021_2025\data\raw\events.csv

<a id="s5"></a>
## 5. Audit depth dan cakupan

Pastikan unduhan mencakup semua bulan dan bandingkan depth dengan geometry_z. Jangan mengubah tanda secara otomatis tanpa definisi sumber.

In [5]:
display(raw[["eventid","depth","geometry_z","depth_source"]].head(10))
assert coverage["complete"].all()
assert raw["depth_source"].eq("properties.depth").all()
print("Manifest:", ROOT / "data/raw/collection_manifest.json")

,eventid,depth,geometry_z,depth_source
0,20210101_0000008,16.0,-16.0,properties.depth
1,20210101_0000007,66.0,-66.0,properties.depth
2,20210101_0000028,10.0,-10.0,properties.depth
3,20210101_0000036,73.0,-73.0,properties.depth
4,20210101_0000035,10.0,-10.0,properties.depth
5,20210101_0000257,34.0,-34.0,properties.depth
6,20210101_0000045,55.0,-55.0,properties.depth
7,20210101_0000055,46.0,-46.0,properties.depth
8,20210101_0000064,48.0,-48.0,properties.depth
9,20210101_0000061,10.0,-10.0,properties.depth


Manifest: C:\Users\yudhi\Documents\machine-learning\seismic_research_2021_2025\data\raw\collection_manifest.json


<a id="s6"></a>
## 6. Jumlah per tahun, jenis catatan, dan rincian bulanan

Tabel ini menjelaskan dari tahun mana data diperoleh. Jumlah per tahun di sini masih sebelum cleaning dan seleksi magnitudo.

In [6]:
annual_raw = raw.assign(tahun=waktu.dt.year).groupby("tahun").size().rename("Catatan mentah").reset_index()
annual_raw["Peran"] = annual_raw["tahun"].map(lambda tahun: "Training" if tahun < 2026 else "Testing (tahun belum lengkap)")
display(annual_raw.rename(columns={"tahun": "Tahun"}))
display(Markdown("### Jenis catatan dari sumber"))
display(raw["event_type"].fillna("Tidak tersedia").value_counts().rename_axis("Kode jenis sumber").reset_index(name="Jumlah catatan"))
display(Markdown("Kode jenis berasal dari EMSC. Hanya jenis yang diizinkan pada konfigurasi yang diteruskan oleh preprocessing; jumlah mentah tidak langsung dianggap jumlah gempa yang dipakai model."))
display(Markdown("### Rincian pengambilan per bulan"))
display(coverage[["start", "end_exclusive", "events_returned", "complete"]].rename(columns={
    "start": "Mulai UTC", "end_exclusive": "Sebelum UTC", "events_returned": "Catatan diambil", "complete": "Lengkap"}))
assert annual_raw["Catatan mentah"].sum() == len(raw)
assert coverage["events_returned"].sum() == len(raw)


,Tahun,Catatan mentah,Peran
0,2021,9148,Training
1,2022,9248,Training
2,2023,10118,Training
3,2024,7188,Training
4,2025,14922,Training
5,2026,27641,Testing (tahun belum lengkap)


### Jenis catatan dari sumber

,Kode jenis sumber,Jumlah catatan
0,ke,78213
1,ue,44
2,uk,8


Kode jenis berasal dari EMSC. Hanya jenis yang diizinkan pada konfigurasi yang diteruskan oleh preprocessing; jumlah mentah tidak langsung dianggap jumlah gempa yang dipakai model.

### Rincian pengambilan per bulan

,Mulai UTC,Sebelum UTC,Catatan diambil,Lengkap
0,2021-01-01T00:00:00+00:00,2021-02-01T00:00:00+00:00,637,True
1,2021-02-01T00:00:00+00:00,2021-03-01T00:00:00+00:00,521,True
2,2021-03-01T00:00:00+00:00,2021-04-01T00:00:00+00:00,771,True
3,2021-04-01T00:00:00+00:00,2021-05-01T00:00:00+00:00,703,True
4,2021-05-01T00:00:00+00:00,2021-06-01T00:00:00+00:00,816,True
...,...,...,...,...
65,2026-06-01T00:00:00+00:00,2026-07-01T00:00:00+00:00,3730,True
66,2026-07-01T00:00:00+00:00,2026-08-01T00:00:00+00:00,2874,True
67,2026-08-01T00:00:00+00:00,2026-09-01T00:00:00+00:00,5147,True
68,2026-09-01T00:00:00+00:00,2026-10-01T00:00:00+00:00,1432,True


## Referensi dan batas interpretasi

[Dokumentasi EMSC](https://www.seismicportal.eu/fdsn-wsevent.html), [clustering scikit-learn](https://scikit-learn.org/stable/modules/clustering.html), dan docs/metodologi.md. Hasil adalah profil aktivitas katalog dalam bounding box dan periode yang dipilih. Interpretasi risiko, prediksi kejadian gempa, serta klaim perubahan fisik memerlukan bukti tambahan.